# MetaTrader5 Docker - Google Colab Deployment
#
# **Testing only** — Colab sessions are ephemeral (max ~12 hours).
#
# ## How this works
#
# Colab runs inside Google's own container. Nested container *builds* are
# blocked (no cgroup writes, no /dev remounting). So the workflow is split:
#
#   1. **Build once** — locally or via GitHub Actions → push to Docker Hub
#   2. **Run in Colab** — pull the pre-built image with udocker (no daemon,
#      no namespaces, no cgroups — uses fakechroot under the hood)
#   3. **Expose** the VNC/web port via ngrok
#
# ## One-time setup: push your image
# Run this locally (once) before using this notebook:
#
#   cd /path/to/MetaTrader5-Docker
#   docker build -t YOUR_DOCKERHUB_USER/mt5-docker:latest .
#   docker push YOUR_DOCKERHUB_USER/mt5-docker:latest
#
# Then set DOCKER_IMAGE below to  YOUR_DOCKERHUB_USER/mt5-docker:latest

## Step 1 — Config (edit this before running)

In [ ]:
# ── SET THESE ─────────────────────────────────────────────────────────────────
VNC_PORT    = 5900   # VNC port
NGROK_TOKEN = ''     # optional: https://dashboard.ngrok.com (free)
# ──────────────────────────────────────────────────────────────────────────────
print(f'VNC port : {VNC_PORT}')

In [ ]:
import subprocess, time, sys, os
from IPython.display import display, HTML

def log(msg): print(msg, flush=True)
def ok(msg):  display(HTML(f'<b style="color:green">✓ {msg}</b>'))
def err(msg): display(HTML(f'<b style="color:red">✗ {msg}</b>')); raise SystemExit(msg)
def run(cmd, **kw): return subprocess.run(cmd, shell=True, **kw)

# ── 1. Install Wine + virtual display + VNC ───────────────────────────────────
log('Installing Wine, Xvfb, x11vnc...')
run('dpkg --add-architecture i386', capture_output=True)
run('apt-get update -qq')
run('apt-get install -y -qq wine wine32 xvfb x11vnc wget')
ok('Wine + VNC installed')

# ── 2. Download MT5 installer ─────────────────────────────────────────────────
MT5_EXE = '/tmp/mt5setup.exe'
if not os.path.exists(MT5_EXE):
    log('Downloading MT5...')
    run(f'wget -q -O {MT5_EXE} https://download.mql5.com/cdn/web/metaquotes.software.corp/mt5/mt5setup.exe')
ok('MT5 installer ready')

# ── 3. Start virtual display (Xvfb) ──────────────────────────────────────────
log('Starting virtual display...')
xvfb = subprocess.Popen(['Xvfb', ':0', '-screen', '0', '1280x720x16'])
time.sleep(2)
os.environ['DISPLAY'] = ':0'
ok('Virtual display running')

# ── 4. Install MT5 via Wine ───────────────────────────────────────────────────
log('Installing MT5 (this takes ~2 min)...')
run(f'DISPLAY=:0 wine {MT5_EXE} /auto', capture_output=True)
time.sleep(30)

# Find the MT5 executable
mt5_bin = None
for path in [
    os.path.expanduser('~/.wine/drive_c/Program Files/MetaTrader 5/terminal64.exe'),
    os.path.expanduser('~/.wine/drive_c/Program Files (x86)/MetaTrader 5/terminal64.exe'),
]:
    if os.path.exists(path):
        mt5_bin = path
        break

if not mt5_bin:
    # Try to find it
    r = run('find ~/.wine -name "terminal64.exe" 2>/dev/null | head -1',
            capture_output=True, text=True)
    mt5_bin = r.stdout.strip()

if mt5_bin:
    ok(f'MT5 installed at: {mt5_bin}')
else:
    err('MT5 binary not found — installer may have failed')

# ── 5. Launch MT5 ─────────────────────────────────────────────────────────────
log('Launching MT5...')
mt5_proc = subprocess.Popen(['wine', mt5_bin], env={**os.environ, 'DISPLAY': ':0'})
time.sleep(5)
ok(f'MT5 running (PID {mt5_proc.pid})')

# ── 6. Start VNC server ────────────────────────────────────────────────────────
log(f'Starting VNC on port {VNC_PORT}...')
vnc_proc = subprocess.Popen(
    ['x11vnc', '-display', ':0', '-forever', '-nopw', '-rfbport', str(VNC_PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(2)
ok(f'VNC server running on port {VNC_PORT}')

# ── 7. Expose via ngrok ───────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyngrok', '-q'], check=True)
from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(VNC_PORT, 'tcp')
ok(f'Connect VNC client to: {public_url}')

In [ ]:
# Run to stop everything
from pyngrok import ngrok
ngrok.kill()
for p in [mt5_proc, vnc_proc, xvfb]:
    p.terminate()
print('Stopped.')

In [ ]:
# Print recent output from the running container process
import select, sys

if run_proc.poll() is None:
    # Non-blocking read of available output
    lines = []
    while True:
        r, _, _ = select.select([run_proc.stdout], [], [], 0.1)
        if not r:
            break
        line = run_proc.stdout.readline()
        if not line:
            break
        lines.append(line.rstrip())
    print('\n'.join(lines[-50:]) if lines else '(no new output yet — container may still be starting)')
else:
    print(f'Container has exited with code {run_proc.returncode}')